# TECH2 Term paper: Boom, slump, recovery? Green energy innovation in European patent data

**Candidate numbers:** XXX, XXX, XXX

## Task 1: Import and prepare the data

The three data files are large (19.7M, 35.6M and 5.1M rows), so we never keep a full file in memory. Each file is read inside its own function, which
- reads **only the columns we need** (and gives them compact data types),
- filters out the rows we don't need, and
- returns only the remaining rows.

The large CPC and applicant files are read in **chunks**, so only a few million rows are in memory at a time.

The order matters. We first build the sample (patents with priority year 2000–2019) and then use its `appln_id`s to filter the other two files.

In [1]:
# Imports
import pandas as pd

In [2]:
# Constants used throughout the notebook

# The data folder must sit next to the notebook (relative path, original file names)
DATA_PATH = "data"
IPC_FILE = f"{DATA_PATH}/202608_EPO_IPC.txt"
CPC_FILE = f"{DATA_PATH}/202608_CPC_Classes.txt"
APP_FILE = f"{DATA_PATH}/202608_EPO_App_reg.txt"

# Sample period (later years are incomplete because of publication delays)
FIRST_YEAR, LAST_YEAR = 2000, 2019

# Number of rows read at a time from the large files
CHUNK_SIZE = 2_000_000

# Table 1: main groups of Y02E (the two digits before the slash)
MAIN_GROUPS = {
    10: "Renewable energy",
    20: "Efficient combustion",
    30: "Nuclear energy",
    40: "Efficient power transmission",
    50: "Non-fossil fuels",
    60: "Storage and hydrogen",
    70: "Other energy conversion",
}

# Table 2: technologies within main group 10 (first digit after the slash)
TECHNOLOGIES = {
    4: "Solar", 5: "Solar", 6: "Solar",
    7: "Wind",
    1: "Other renewables", 2: "Other renewables", 3: "Other renewables",
}

In [3]:
def read_sample(path, first_year, last_year):
    """Return one row per EPO patent application with priority year in [first_year, last_year].

    The IPC file has one row per IPC code, so an application appears several times.
    We only need appln_id and prio_year, and keep one row per application.
    """
    df = pd.read_csv(
        path,
        sep="|",
        usecols=["appln_id", "prio_year"],
        dtype={"appln_id": "int32", "prio_year": "int16"},
    )
    df = df[df["prio_year"].between(first_year, last_year)]
    return df.drop_duplicates("appln_id").reset_index(drop=True)

In [4]:
def read_green_codes(path, sample_ids):
    """Return all Y02E codes of the patents in the sample, one row per (appln_id, code).

    The file is read in chunks, and from each chunk we keep only Y02E codes of
    applications in the sample, so the 35.6M-row file is never fully in memory.
    """
    parts = []
    reader = pd.read_csv(
        path,
        sep="|",
        usecols=["appln_id", "CPC_Class"],
        dtype={"appln_id": "int32", "CPC_Class": "str"},
        chunksize=CHUNK_SIZE,
    )
    for chunk in reader:
        keep = chunk["CPC_Class"].str.startswith("Y02E") & chunk["appln_id"].isin(sample_ids)
        parts.append(chunk[keep])
    return pd.concat(parts, ignore_index=True)

In [5]:
def read_applicants(path, sample_ids):
    """Return the applicants of the patents in the sample.

    - Applicants linked to several regions appear on several otherwise identical
      rows. Since we drop the region columns, these rows become exact duplicates,
      and we drop them so that each applicant is counted once.
    - Names are converted to upper case so that names differing only in
      capitalization are treated as the same applicant (needed in Task 5).
    """
    parts = []
    reader = pd.read_csv(
        path,
        sep="|",
        encoding="utf-8-sig",  # the file starts with a byte order mark
        usecols=["appln_id", "app_name", "ctry_code", "app_share"],
        dtype={"appln_id": "int32", "app_share": "float64"},
        chunksize=CHUNK_SIZE,
    )
    for chunk in reader:
        parts.append(chunk[chunk["appln_id"].isin(sample_ids)])
    df = pd.concat(parts, ignore_index=True).drop_duplicates(ignore_index=True)
    df["app_name"] = df["app_name"].str.upper()
    df["ctry_code"] = df["ctry_code"].astype("category")
    return df

In [6]:
def classify_codes(codes):
    """Add the main group (Table 1) and technology (Table 2) of each Y02E code.

    Codes look like 'Y02E  10/549': the two digits before the slash give the
    main group, and in main group 10 the first digit after the slash gives the
    technology. Codes in main group 10 not listed in Table 2 get no technology (NaN).
    """
    parts = codes["CPC_Class"].str.extract(r"Y02E\s*(\d{2})/(\d)").astype("int8")
    codes = codes.assign(
        main_group=parts[0].map(MAIN_GROUPS),
        technology=parts[1].map(TECHNOLOGIES).where(parts[0] == 10),
    )
    return codes

In [7]:
def build_categories(codes, sample):
    """Return one row per (green patent, category) with the patent's priority year.

    A category is either a main group (Table 1) or a technology (Table 2).
    A patent counts once per category, however many of its codes fall in it,
    so duplicates are dropped. The 'level' column tells the two kinds apart.
    """
    long = pd.concat([
        codes[["appln_id", "main_group"]]
            .rename(columns={"main_group": "category"})
            .assign(level="main group"),
        codes[["appln_id", "technology"]]
            .dropna()
            .rename(columns={"technology": "category"})
            .assign(level="technology"),
    ])
    long = long.drop_duplicates(["appln_id", "category"]).merge(sample, on="appln_id")
    return long.astype({"category": "category", "level": "category"}).reset_index(drop=True)

### Load the data

The three files are read once here. Tasks 2–5 only use the DataFrames created in Task 1:

| DataFrame | One row per | Columns |
|---|---|---|
| `sample` | patent in the sample | `appln_id`, `prio_year` |
| `green_codes` | Y02E code of a green patent | `appln_id`, `CPC_Class`, `main_group`, `technology` |
| `green_categories` | (green patent, category) | `appln_id`, `category`, `level`, `prio_year` |
| `applicants` | applicant of a green patent | `appln_id`, `app_name`, `ctry_code`, `app_share` |

In [8]:
# 1) The sample: all EPO applications with priority year 2000-2019
sample = read_sample(IPC_FILE, FIRST_YEAR, LAST_YEAR)

# 2) Y02E codes of patents in the sample, with main group and technology
green_codes = classify_codes(read_green_codes(CPC_FILE, sample["appln_id"]))

# 3) Applicants of patents in the sample
applicants = read_applicants(APP_FILE, sample["appln_id"])

# 4) One row per (green patent, category), with priority year
green_categories = build_categories(green_codes, sample)

### Checkpoints

In [9]:
n_green = green_codes["appln_id"].nunique()

checkpoints = pd.DataFrame({
    "Our result": [
        len(sample),
        n_green,
        round(applicants["app_share"].sum()),
    ],
    "Expected": [2_835_599, 114_924, 2_835_140],
}, index=[
    "Patent applications in the sample",
    "Green energy patents in the sample",
    "Sum of app_share in the sample",
])
checkpoints["Match"] = checkpoints["Our result"] == checkpoints["Expected"]
checkpoints

,Our result,Expected,Match
Patent applications in the sample,2835599,2835599,True
Green energy patents in the sample,114924,114924,True
Sum of app_share in the sample,2835140,2835140,True


### Overview of the data structure

In [10]:
# Average number of Y02E codes per green patent (green_codes has one row per code)
avg_codes = len(green_codes) / n_green

# Green patents in more than one main group
groups_per_patent = green_codes.groupby("appln_id")["main_group"].nunique()
n_multi_group = (groups_per_patent > 1).sum()

# Patents in the sample with applicants in more than one country
countries_per_patent = applicants.groupby("appln_id")["ctry_code"].nunique()
n_multi_country = (countries_per_patent > 1).sum()

pd.DataFrame({"Value": [
    f"{avg_codes:.2f}",
    f"{n_multi_group:,}",
    f"{n_multi_country:,}",
]}, index=[
    "Average number of Y02E codes per green patent",
    "Green patents in more than one main group",
    "Patents with applicants in more than one country",
])

,Value
Average number of Y02E codes per green patent,1.23
Green patents in more than one main group,"4,359"
Patents with applicants in more than one country,"58,954"


In [11]:
# Tasks 2-5 only need the applicants of green patents. Now that the checkpoints
# that use all applicants are done, we keep only those rows to free memory.
applicants = applicants[applicants["appln_id"].isin(green_codes["appln_id"])].reset_index(drop=True)
print(f"Applicant rows kept for green patents: {len(applicants):,}")

Applicant rows kept for green patents: 126,097


**Results.** All three checkpoints match. The overview numbers show how the data are structured:

- A green patent has on average **1.23** Y02E codes. Most have only one, but some have several, which is why we count each patent once per category instead of counting codes.
- **4,359** green patents belong to more than one main group, so the main groups overlap. The counts per main group therefore add up to more than the number of green patents.
- **58,954** patents in the sample have applicants in more than one country. This is why Task 5 uses fractional counting (`app_share`), so that such patents are not counted in full for every country.